# Megaline Mobile Plan Classification
Portfolio copy of TripleTen coursework. Predict the recorded Smart/Ultra plan label from calls, minutes, messages, and data usage. Numeric outputs are retained from the supplied notebook; portable-path and narrative edits were not rerun because the CSV was not supplied.


In this project, the goal is to build a machine learning model that can recommend the most suitable mobile plan for Megaline subscribers based on their monthly usage behavior. Using historical data from customers who have already switched to newer plans, the task is framed as a classification problem where the model predicts whether a user should be offered the Smart or Ultra plan. Several models are trained and evaluated to identify the approach that delivers the highest accuracy while maintaining reliable performance on unseen data.

In [1]:
import pandas as pd
from IPython.display import display

from pathlib import Path
import os
data_path = Path(os.environ.get('MEGALINE_DATA_PATH', 'data/users_behavior.csv'))
if not data_path.exists() and Path('/datasets/users_behavior.csv').exists():
    data_path = Path('/datasets/users_behavior.csv')
if not data_path.exists():
    raise FileNotFoundError('Place users_behavior.csv in data/ or set MEGALINE_DATA_PATH.')
df = pd.read_csv(data_path)

# Preview the data
display(df.head())

# Dataset structure
df.info()

# Class distribution
display(df['is_ultra'].value_counts(normalize=True))



,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


0    0.693528
1    0.306472
Name: is_ultra, dtype: float64

In [2]:
from sklearn.model_selection import train_test_split

features = df.drop('is_ultra', axis=1)
target = df['is_ultra']

# Train + temp
features_train, features_temp, target_train, target_temp = train_test_split(
    features, target, test_size=0.4, random_state=42, stratify=target
)

# Validation + test
features_valid, features_test, target_valid, target_test = train_test_split(
    features_temp, target_temp, test_size=0.5, random_state=42, stratify=target_temp
)


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42
)

rf.fit(features_train, target_train)

pred_valid_rf = rf.predict(features_valid)
print("Random Forest validation accuracy:", accuracy_score(target_valid, pred_valid_rf))


Random Forest validation accuracy: 0.8102643856920684


In [4]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    max_depth=6,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)
dt.fit(features_train, target_train)

pred_valid_dt = dt.predict(features_valid)
print("Decision Tree accuracy:", accuracy_score(target_valid, pred_valid_dt))


Decision Tree accuracy: 0.7729393468118196


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# Base model
rf = RandomForestClassifier(random_state=42)




# Hyperparameter grid to test
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [8, 10, 12, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}


# Grid search
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)

grid_search.fit(features_train, target_train)

# Best model from tuning
best_rf = grid_search.best_estimator_

print("Best parameters found:")
print(grid_search.best_params_)



# Evaluate the best model on validation set
pred_valid_best = best_rf.predict(features_valid)
best_accuracy = accuracy_score(target_valid, pred_valid_best)

print(f"\nValidation accuracy comparison:")
print(f"Original Random Forest: 0.8103")
print(f"Tuned Random Forest: {best_accuracy:.4f}")
print(f"Improvement: {best_accuracy - 0.8103:.4f}")

Best parameters found:
{'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}

Validation accuracy comparison:
Original Random Forest: 0.8103
Tuned Random Forest: 0.8072
Improvement: -0.0031


In [6]:
# Combine train + validation
features_train_full = pd.concat([features_train, features_valid])
target_train_full = pd.concat([target_train, target_valid])

rf_final = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42
)

rf_final.fit(features_train_full, target_train_full)

pred_test = rf_final.predict(features_test)
test_accuracy = accuracy_score(target_test, pred_test)

print("=== FINAL TEST ACCURACY ===")
print(f"Accuracy: {test_accuracy:.2%}")


=== FINAL TEST ACCURACY ===
Accuracy: 81.65%


## Model comparison
The original Random Forest validation accuracy was 0.8103; the tuned forest scored 0.8072, so tuning did not improve validation performance. The original configuration was refit on training plus validation before test evaluation. Later cells also evaluate Logistic Regression and Decision Tree on the same test split, so the test results should be treated as exploratory comparisons rather than a fully untouched final holdout.

In [7]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

logreg_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, random_state=42))
])

logreg_param_grid = {
    "clf__C": [0.01, 0.1, 1, 10],
    "clf__penalty": ["l2"],
    "clf__solver": ["lbfgs"]
}

logreg_search = GridSearchCV(
    logreg_pipe,
    logreg_param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1
)

logreg_search.fit(features_train, target_train)

best_logreg = logreg_search.best_estimator_
pred_test_logreg = best_logreg.predict(features_test)

print("LOGISTIC REGRESSION")
print("Best params:", logreg_search.best_params_)
print("Best CV accuracy:", logreg_search.best_score_)
print("Test accuracy:", accuracy_score(target_test, pred_test_logreg))
print(classification_report(target_test, pred_test_logreg))


LOGISTIC REGRESSION
Best params: {'clf__C': 1, 'clf__penalty': 'l2', 'clf__solver': 'lbfgs'}
Best CV accuracy: 0.7416916761994482
Test accuracy: 0.7325038880248833
              precision    recall  f1-score   support

           0       0.73      0.98      0.84       446
           1       0.78      0.18      0.29       197

    accuracy                           0.73       643
   macro avg       0.75      0.58      0.56       643
weighted avg       0.74      0.73      0.67       643



In [8]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)

dt_param_grid = {
    "max_depth": [3, 5, 8, 10, 12, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
    "criterion": ["gini", "entropy"]
}

dt_search = GridSearchCV(
    dt,
    dt_param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1
)

dt_search.fit(features_train, target_train)

best_dt = dt_search.best_estimator_
pred_test_dt = best_dt.predict(features_test)

print("\nDECISION TREE")
print("Best params:", dt_search.best_params_)
print("Best CV accuracy:", dt_search.best_score_)
print("Test accuracy:", accuracy_score(target_test, pred_test_dt))
print(classification_report(target_test, pred_test_dt))



DECISION TREE
Best params: {'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 8, 'min_samples_split': 20}
Best CV accuracy: 0.7972074557566785
Test accuracy: 0.7916018662519441
              precision    recall  f1-score   support

           0       0.80      0.94      0.86       446
           1       0.77      0.45      0.57       197

    accuracy                           0.79       643
   macro avg       0.78      0.70      0.72       643
weighted avg       0.79      0.79      0.77       643



In [9]:
# Baseline: predict majority class
baseline_accuracy = target_test.value_counts(normalize=True).max()
print("Baseline accuracy:", baseline_accuracy)


Baseline accuracy: 0.6936236391912908


## Conclusion
The saved Random Forest test accuracy is **81.65%**, versus a majority-class baseline of **69.36%**, a difference of approximately **12.29 percentage points** on 643 test records. Logistic Regression and Decision Tree test accuracies were 73.25% and 79.16%. These are saved notebook results, not recomputed here. Predicting the recorded Smart/Ultra label does not establish which plan is optimal for a customer or demonstrate business lift.